# 📝 [제 11편] MNIST - 손글씨 숫자를 읽는 신경망

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「Deep Learning (IV) - MNIST (필기체숫자) 인식」 (ML_LEC_22)

---

## 🎯 학습 목표
1. MNIST 데이터의 구조(정답 1개 + 픽셀 784개)를 이해하고 이미지로 나타낼 수 있다.
2. 입력 정규화와 원-핫 인코딩(one-hot encoding)의 의미를 설명할 수 있다.
3. 784 - 100 - 10 신경망의 구조를 알고, 수치미분으로 학습할 때의 문제점(시간)을 직접 확인한다.

---

## 🧭 지난 시간 복습
* 제 10편: 입력층 → 은닉층 → 출력층 으로 피드포워드하고, 손실을 줄이도록 모든 W, b 를 학습시켜 XOR 을 풀었습니다.
* 오늘은 같은 방법을 **진짜 손글씨 숫자 사진** 에 적용합니다.

[⬅ 이전 : 제 10편 신경망과 딥러닝 - 피드포워드로 XOR 풀기](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/10_neural_network.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/00_index.ipynb)　|　[다음 : 제 12편 오차역전파 (1) - 연쇄법칙으로 한꺼번에 미분하기 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/12_backprop_1.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

In [ ]:
#@title ▶ 수치미분, sigmoid 함수 (실행만 하세요)
def numerical_derivative(f, x):
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x; fx1 = f(x)
        x[idx] = float(tmp_val) - delta_x; fx2 = f(x)
        grad[idx] = (fx1 - fx2) / (2 * delta_x)
        x[idx] = tmp_val
        it.iternext()
    return grad

def sigmoid(z):
    return 1 / (1 + np.exp(-z))
print('준비 완료')

## 📖 개념 다지기 1. MNIST 란?

* 사람이 손으로 쓴 숫자 0 ~ 9 의 흑백 사진 모음 (학습용 **60,000** 장, 테스트용 **10,000** 장)
* 프로그래밍을 배울 때 처음 출력하는 'Hello, World' 처럼, **딥러닝을 배우면 처음 해 보는 데이터**
* 사진 1장 = 가로 28 × 세로 28 = ( 　　 ) 개의 숫자 (0 = 검정 ~ 255 = 흰색)

원본 강의의 mnist_train.csv 는 한 줄에 숫자가 **785 개** 있습니다.

| 1열 | 2열 ~ 785열 |
|:-:|:-:|
| 정답 (예: 5) | 숫자 5 의 사진을 이루는 픽셀 값 784 개 |

In [ ]:
#@title ▶ MNIST 불러오기 (원본 강의와 같은 '정답 1열 + 픽셀 784열' 모양으로 만듭니다)
try:
    from tensorflow.keras.datasets import mnist
    (x_tr, t_tr), (x_te, t_te) = mnist.load_data()
    training_data = np.c_[t_tr, x_tr.reshape(-1, 784)].astype(np.float64)
    test_data     = np.c_[t_te, x_te.reshape(-1, 784)].astype(np.float64)
    print('✅ MNIST 불러오기 성공')
except Exception as e:
    # 인터넷이나 tensorflow 가 없을 때: 8 × 8 크기의 작은 손글씨 숫자 데이터로 대신합니다
    from sklearn.datasets import load_digits
    d = load_digits()
    X = d.images.reshape(-1, 64) * (255 / 16)
    data = np.c_[d.target, X]
    training_data, test_data = data[:1500], data[1500:]
    print('⚠️ MNIST 를 불러오지 못해 8 × 8 손글씨 데이터로 대신합니다:', e)

IMG = int(np.sqrt(training_data.shape[1] - 1))      # 사진 한 변의 픽셀 수 (MNIST 는 28)
print('training_data.shape =', training_data.shape, ', test_data.shape =', test_data.shape)
print('첫 번째 데이터의 정답 =', int(training_data[0, 0]))

> **[확인 1]** training_data 의 형상은 ( 　　 × 　　 ) 이다. 행 하나 = 사진 ( 　 ) 장이다.

## 🎬 애니메이션 1. 784 개의 숫자가 사진이 되는 과정

한 줄로 늘어선 784 개의 픽셀 값을 28 개씩 끊어서 한 줄씩 쌓으면 (`reshape(28, 28)`) 사진이 됩니다.

In [ ]:
#@title 🎬 애니메이션 1 실행 (reshape)
row = training_data[0]
label, pixels = int(row[0]), row[1:]
img = pixels.reshape(IMG, IMG)
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'width_ratios': [1, 1.1]})
frames = list(range(IMG + 1)) + [IMG]*8
def draw(k):
    axL.clear(); axR.clear()
    show = np.full((IMG, IMG), np.nan); show[:k] = img[:k]
    axL.imshow(show, cmap='gray', vmin=0, vmax=255)
    axL.set_title(f'{k} / {IMG} 줄 채움   (정답 : {label})', fontsize=13); axL.axis('off')
    seg = pixels[k*IMG:(k+1)*IMG] if k < IMG else pixels[-IMG:]
    axR.axis('off')
    axR.text(0, 0.95, f'{k+1 if k < IMG else IMG}번째 줄의 픽셀 값 {IMG}개 :', fontsize=12, transform=axR.transAxes)
    txt = '\n'.join(' '.join(f'{int(v):3d}' for v in seg[i:i+7]) for i in range(0, IMG, 7))
    axR.text(0, 0.85, txt, fontsize=12, family='monospace', va='top', transform=axR.transAxes)
    axR.text(0, 0.25, f'전체 {IMG*IMG}개 = {IMG}개 × {IMG}줄\npixels.reshape({IMG}, {IMG})', fontsize=13, transform=axR.transAxes, color='crimson')
anim = animation.FuncAnimation(fig, draw, frames=frames, interval=150)
plt.close(fig)
show_anim(anim)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(13, 3.6))
for ax, row in zip(axes.ravel(), training_data[:16]):
    ax.imshow(row[1:].reshape(IMG, IMG), cmap='gray'); ax.set_title(f'정답 {int(row[0])}'); ax.axis('off')
plt.suptitle('학습 데이터 앞의 16장'); plt.tight_layout(); plt.show()

## 📖 개념 다지기 2. 신경망 설계 (784 - 100 - 10)

| 층 | 노드 수 | 이유 |
|:---|:-:|:---|
| 입력층 | 784 | 픽셀 784 개를 하나씩 받는다 |
| 은닉층 | 100 | 정해진 규칙 없음 (임의로 100) |
| 출력층 | 10 | 정답이 0 ~ 9 중 하나이므로 숫자마다 노드 하나 |

　W2 : (784 × 100),  b2 : 100,  W3 : (100 × 10),  b3 : 10  → 학습할 값이 무려 ( 　　　　 ) 개!

### 원-핫 인코딩 (one-hot encoding)
정답 5 를 출력 노드 10 개에 맞춰 **5 번 자리만 크고 나머지는 작은** 리스트로 바꿉니다. (sigmoid 출력에 맞춰 0, 1 대신 0.01, 0.99 사용)

| 인덱스 | 0 | 1 | 2 | 3 | 4 | **5** | 6 | 7 | 8 | 9 |
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| 정답 5 | 0.01 | 0.01 | 0.01 | 0.01 | 0.01 | **0.99** | 0.01 | 0.01 | 0.01 | 0.01 |

예측할 때는 출력 10 개 중 **가장 큰 값의 위치** (`np.argmax`) 를 답으로 고릅니다.

### 입력 정규화 (normalize)
픽셀 값 0 ~ 255 를 그대로 넣으면 e<sup>−z</sup> 계산에서 너무 큰 수가 나와 문제(overflow)가 생길 수 있습니다.
→ **(픽셀 / 255) × 0.99 + 0.01** 로 0.01 ~ 1 사이 값으로 바꿉니다.

> **[연습 1]** 정답 3 의 원-핫 인코딩: [ 0.01, 0.01, 0.01, ( 　 ), 0.01, … ]
> 픽셀 값 255 를 정규화하면 ( 　　 ), 0 을 정규화하면 ( 　　 )

In [ ]:
label = 5
target = np.zeros(10) + 0.01
target[label] = 0.99
print('정답 5 의 원-핫 인코딩 :', target)
print('np.argmax(target) =', np.argmax(target))

pix = np.array([0, 128, 255])
print('정규화 :', pix, '→', (pix / 255.0 * 0.99) + 0.01)

## 💻 실습 1. NeuralNetwork 클래스 (수치미분 버전, 원본 강의 코드)

In [ ]:
class NeuralNetwork:
    def __init__(self, input_nodes, hidden_nodes, output_nodes):
        self.input_nodes = input_nodes          # 784
        self.hidden_nodes = hidden_nodes        # 100
        self.output_nodes = output_nodes        # 10
        self.W2 = np.random.rand(self.input_nodes, self.hidden_nodes)    # 입력층-은닉층 가중치
        self.b2 = np.random.rand(self.hidden_nodes)                      # 은닉층 바이어스
        self.W3 = np.random.rand(self.hidden_nodes, self.output_nodes)   # 은닉층-출력층 가중치
        self.b3 = np.random.rand(self.output_nodes)                      # 출력층 바이어스
        self.learning_rate = 1e-4

    def feed_forward(self):                     # 피드포워드 → 손실 (cross-entropy)
        delta = 1e-7
        z1 = np.dot(self.input_data, self.W2) + self.b2
        y1 = sigmoid(z1)
        z2 = np.dot(y1, self.W3) + self.b3
        y = sigmoid(z2)
        return -np.sum(self.target_data*np.log(y + delta) + (1 - self.target_data)*np.log((1 - y) + delta))

    def loss_val(self):
        return self.feed_forward()

    def train(self, training_data):             # 사진 1장으로 1번 학습
        self.target_data = np.zeros(self.output_nodes) + 0.01        # 원-핫 인코딩
        self.target_data[int(training_data[0])] = 0.99
        self.input_data = (training_data[1:] / 255.0 * 0.99) + 0.01  # 정규화
        f = lambda x : self.feed_forward()
        self.W2 -= self.learning_rate * numerical_derivative(f, self.W2)
        self.b2 -= self.learning_rate * numerical_derivative(f, self.b2)
        self.W3 -= self.learning_rate * numerical_derivative(f, self.W3)
        self.b3 -= self.learning_rate * numerical_derivative(f, self.b3)

    def predict(self, input_data):
        z1 = np.dot(input_data, self.W2) + self.b2
        y1 = sigmoid(z1)
        z2 = np.dot(y1, self.W3) + self.b3
        y = sigmoid(z2)
        return np.argmax(y)                     # 가장 큰 값의 위치 = 예측한 숫자

    def accuracy(self, test_data):
        matched = 0
        for row in test_data:
            label = int(row[0])
            data = (row[1:] / 255.0 * 0.99) + 0.01
            if self.predict(data) == label:
                matched += 1
        return matched / len(test_data)

print('NeuralNetwork 클래스 준비 완료')

## ⏱️ 실습 2. 수치미분으로 학습하면 얼마나 걸릴까?

사진 **딱 1장** 으로 1번 학습하는 데 걸리는 시간을 재고, 원본 강의처럼 30,000장을 학습하면 몇 시간이 걸릴지 계산해 봅시다.
(10초 안팎이 걸립니다. 기다리는 동안 아래 질문을 생각해 보세요)

> 학습할 값이 약 8만 개이고, 값 하나를 수치미분할 때마다 피드포워드를 ( 　 ) 번 해야 하므로, 사진 1장에 피드포워드를 약 ( 　　　 ) 번 해야 합니다.

In [ ]:
import time
np.random.seed(0)
nn = NeuralNetwork(IMG*IMG, 100, 10)
n_params = nn.W2.size + nn.b2.size + nn.W3.size + nn.b3.size
print('학습할 값의 개수 =', n_params)

start = time.time()
nn.train(training_data[0])          # 사진 1장으로 1번 학습
sec = time.time() - start
print(f'사진 1장 학습 시간 = {sec:.2f} 초')
print(f'→ 30,000장 학습 예상 시간 = {sec*30000/3600:.1f} 시간  (= {sec*30000/3600/24:.1f} 일)')

> **[확인 2]** 사진 1장 학습에 ( 　 ) 초, 30,000장이면 약 ( 　 ) 시간이 걸린다.
> 원본 강의에서는 이렇게 오래 학습해서 정확도 **94.34 %** 를 얻었습니다.

💡 **수치미분의 문제점**: 직관적이지만 값 하나하나를 따로 미분하므로 **너무 느리다!**
→ 해결 방법: 미분을 **수학 공식(체인 룰)으로 한꺼번에** 계산하는 **오차역전파 (Back Propagation)** → 제 12 ~ 13편

## 🚀 미리보기. 오차역전파로 학습하면?

아래 셀은 **제 13편에서 배울 오차역전파 공식** 으로 같은 784 - 100 - 10 신경망을 학습시킵니다. (지금은 실행만 해 보세요. 1분 정도)

In [ ]:
#@title 🚀 오차역전파 미리보기 (코드는 제 13편에서 배웁니다)
class NeuralNetworkBP:
    def __init__(self, i_nodes, h_nodes, o_nodes, learning_rate):
        self.W2 = np.random.randn(i_nodes, h_nodes) / np.sqrt(i_nodes / 2)
        self.b2 = np.random.rand(h_nodes)
        self.W3 = np.random.randn(h_nodes, o_nodes) / np.sqrt(h_nodes / 2)
        self.b3 = np.random.rand(o_nodes)
        self.lr = learning_rate
    def forward(self, x):
        a2 = sigmoid(x @ self.W2 + self.b2)
        return a2, sigmoid(a2 @ self.W3 + self.b3)
    def train(self, x, t):
        x = x.reshape(1, -1); t = t.reshape(1, -1)
        a2, a3 = self.forward(x)
        loss_3 = (a3 - t) * a3 * (1 - a3)
        loss_2 = (loss_3 @ self.W3.T) * a2 * (1 - a2)
        self.W3 -= self.lr * a2.T @ loss_3; self.b3 -= self.lr * loss_3[0]
        self.W2 -= self.lr * x.T @ loss_2;  self.b2 -= self.lr * loss_2[0]
    def predict(self, x):
        return self.forward(x)[1]

np.random.seed(0)
bp = NeuralNetworkBP(IMG*IMG, 100, 10, 0.3)
X_test = test_data[:, 1:] / 255.0 * 0.99 + 0.01
t_test = test_data[:, 0].astype(int)
epochs = 1 if len(training_data) > 10000 else 15
start = time.time()
for ep in range(epochs):
    for row in training_data:
        t = np.zeros(10) + 0.01; t[int(row[0])] = 0.99
        bp.train(row[1:] / 255.0 * 0.99 + 0.01, t)
acc = np.mean(np.argmax(bp.predict(X_test), axis=1) == t_test)
print(f'학습한 사진 수 = {len(training_data)*epochs},  걸린 시간 = {time.time()-start:.1f} 초')
print(f'테스트 사진 {len(test_data)}장 정확도 = {acc*100:.2f} %')

> **[확인 3]** 오차역전파로는 약 ( 　 ) 초 만에 정확도 ( 　 ) % 를 얻었다. 수치미분과 비교하면 몇 배쯤 빠른가요? ( 　　　 )

## 🎬 애니메이션 2. 신경망이 숫자를 읽는 모습

테스트 사진(학습에 쓰지 않은 사진)을 한 장씩 보여 주고, 출력 노드 10 개의 값을 막대그래프로 나타냅니다. 가장 큰 막대(argmax)가 신경망의 답입니다.

In [ ]:
#@title 🎬 애니메이션 2 실행 (미리보기를 먼저 실행하세요)
pred_all = np.argmax(bp.predict(X_test), axis=1)
wrong = np.where(pred_all != t_test)[0][:3]
idxs = list(range(9)) + list(wrong)              # 앞의 9장 + 틀린 사진 3장
frames = [i for i in idxs for _ in range(6)]
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={'width_ratios': [1, 1.6]})
def draw(f):
    i = frames[f]
    out = bp.predict(X_test[i:i+1])[0]
    p, t = int(np.argmax(out)), t_test[i]
    axL.clear(); axR.clear()
    axL.imshow(test_data[i, 1:].reshape(IMG, IMG), cmap='gray'); axL.axis('off')
    axL.set_title(f'테스트 사진 #{i}   정답 : {t}', fontsize=13)
    colors = ['lightgray']*10; colors[p] = 'green' if p == t else 'crimson'
    axR.bar(range(10), out, color=colors); axR.set_xticks(range(10)); axR.set_ylim(0, 1.05)
    axR.set_xlabel('출력 노드 (숫자)'); axR.set_ylabel('출력 값')
    axR.set_title(f'신경망의 답 : {p}  ' + ('(정답)' if p == t else '(오답)'), fontsize=14, color='green' if p == t else 'crimson')
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=200)
plt.close(fig)
show_anim(anim)

> **[관찰 1]** 신경망이 틀린 사진들은 사람이 봐도 헷갈리는 모양인가요? 어떤 숫자를 어떤 숫자로 착각했나요?
> (                                                                                   )

## ✏️ 연습 문제

> **[문제 1]** 은닉층 노드를 100 개에서 50 개로 줄이면 학습할 값은 몇 개가 되나요? ( 　　　 ) 개

> **[문제 2]** 내가 직접 고른 테스트 사진 번호로 신경망의 답을 확인하는 코드를 완성하세요.

In [ ]:
i = ______                                       # 0 ~ 테스트 사진 수 - 1 사이의 번호
out = bp.predict(X_test[i:i+1])[0]
print('정답 =', t_test[i], ', 신경망의 답 =', ______)
plt.imshow(test_data[i, 1:].reshape(IMG, IMG), cmap='gray'); plt.axis('off'); plt.show()

In [ ]:
#@title 🔑 정답 확인 { display-mode: "form" }
n_in = IMG*IMG
print(f'문제 1 : W2 {n_in}×50 + b2 50 + W3 50×10 + b3 10 = {n_in*50 + 50 + 500 + 10} 개 (MNIST 이면 39,760 개)')
print('문제 2 : 신경망의 답 = np.argmax(out)')

## 📌 정리

| 핵심 | 내용 |
|:---|:---|
| MNIST | 28 × 28 = 784 픽셀 손글씨 숫자, 학습 60,000 / 테스트 10,000 |
| 데이터 모양 | 1열 정답 + 784열 픽셀 → `row[0]`, `row[1:]` |
| 정규화 | (픽셀 / 255) × 0.99 + 0.01 |
| 원-핫 인코딩 | 정답 자리만 0.99, 나머지 0.01 / 예측은 `np.argmax` |
| 신경망 | 784 - 100 - 10, 학습할 값 79,510 개 |
| 문제점 | 수치미분은 너무 느리다 → **오차역전파** |

### 🏆 도전 과제
수치미분은 W 값 8만 개를 **하나씩** 조금씩 움직여 봐야 했습니다. 제 1편에서 배운 **연쇄법칙 (chain rule)** 을 쓰면, 출력층의 오차 하나로부터 모든 W 의 미분값을 한꺼번에 구할 수 있을까요?
→ 다음 시간: **오차역전파**

---

[⬅ 이전 : 제 10편 신경망과 딥러닝 - 피드포워드로 XOR 풀기](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/10_neural_network.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/00_index.ipynb)　|　[다음 : 제 12편 오차역전파 (1) - 연쇄법칙으로 한꺼번에 미분하기 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/12_backprop_1.ipynb)